# Welcome to Week 5 - Agent Frameworks

## Day 2: Pydantic AI

Same week, same five steps, a new framework. The whole idea this week is that once you know one agent framework, you basically know them all, so every day we build the same agent the same five steps and watch the idioms rhyme:

1. **Create the agent** - give it a model and a system prompt.
2. **Run it** - send a message, get a reply.
3. **Add tools** - plain typed functions the agent can call.
4. **Add MCP** - connect a tool server someone else wrote, wired the same way every time.
5. **Put it in a loop with a goal** - hand it an objective and let it work, step after step, until the job is done.

Steps 1 and 2 are still just an LLM call. Tools and MCP give the agent things it can do. Step 5 is where it becomes an agent: the framework runs the loop on its own, choosing a tool, reading the result, choosing again, until the goal is met.

The running project is the same SQLite todo board from Day 1. A worker picks one goal off the board, plans its own steps, does the work with its agent loop, and ticks each step off. The board code (`board.py`) is the same file, byte for byte; only the framework around it changes.

This is the second framework of the day: **Pydantic AI**, from the team behind Pydantic, the validation library under FastAPI and most of the modern Python data stack. It brings that same types-everywhere feel to agents: you declare what a tool takes and what an agent returns, and the framework validates and types the whole flow. It is famous for the cleanest typed-output story in the field and for one-line observability through Logfire. Notice how little changes from Strands; that is the point.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The Pydantic AI docs</h2>
            <span style="color:#00bfff;">The docs live at <a href="https://pydantic.dev/docs/ai/">https://pydantic.dev/docs/ai/</a>. Pydantic AI reached v1 in 2025 and a v2 is in beta, so we pin the version (1.107.0 here) and prefer the current pages over older blog posts; the MCP wiring in particular changed recently.</span>
        </td>
    </tr>
</table>

## Setup

Two things this day needs, both already in place from earlier weeks:

- **Node**, for `npx` (the filesystem MCP server runs over it). Check with `node --version`.
- **`OPENAI_API_KEY`** in the repo-root `.env`. Today's frameworks use OpenAI's `gpt-5.4-mini`, and that key has been in your `.env` since Week 1.

Pydantic AI is part of the repo's environment, so a normal `uv sync` from the repo root installs everything. Open this notebook in Cursor and pick the repo's default **Python 3.12.12** kernel, the same one you use every week, then run the cells top to bottom.

Warm the filesystem MCP server once now so the first run is fast, then stop it with Ctrl-C as soon as it prints that it is running:

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

load_dotenv(override=True)

True

## Step 1: Create the agent

In Pydantic AI an agent is an `Agent`: a model string plus `instructions`, which is its system prompt. The string `"openai-chat:gpt-5.4-mini"` picks the provider and the model, with `OPENAI_API_KEY` coming from the repo-root `.env`. The `-chat` names OpenAI's Chat Completions API explicitly. A bare `openai:` works too, but its default shifts in the coming Pydantic AI v2, and Chat Completions is the API every OpenAI-compatible endpoint speaks, which is what we want for the model swap at the end of the day.

In [2]:
# 사용할 모델 식별자. "openai-chat:" 접두사는 OpenAI 채팅 계열 모델임을 지정
MODEL = "openai-chat:gpt-5.4-mini"

# 에이전트 생성: 모델과 지속 적용되는 시스템 지시문만 지정한 최소 구성
agent = Agent(
    MODEL,
    instructions="당신은 간결하고 친절한 어시스턴트입니다. 짧은 한 문장으로 답하세요.",
)

## Step 2: Run it

Send a message, await the reply, and print the result's `.output`. With no tools yet there is nothing to loop over, so the agent just answers; this is still only an LLM call. There is a synchronous `agent.run_sync(...)` too, but inside a notebook we await `agent.run(...)` so it cooperates with Jupyter's event loop.

In [3]:
# 에이전트 실행: 사용자 메시지를 전달하고 응답이 올 때까지 대기
result = await agent.run("스페인어로 인사말을 해 주세요.")
# 에이전트의 최종 응답 텍스트만 출력(도구 호출 기록 등은 result의 다른 속성에 있음)
print(result.output)

¡Hola!


## Our project this week: a SQLite todo board

The worker coordinates through the same tiny SQLite board as Day 1, the same `board.py` file: one file, one table, no server to run. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [4]:
import board  # 할 일 보드 조작 모듈

board.reset_board()  # 보드의 기존 항목을 모두 제거해 빈 상태로 초기화
# 목표를 보드에 등록. 이 문자열은 에이전트가 읽어서 수행할 작업 지시문이다
board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.")
board.list_todos()  # 현재 보드에 있는 모든 할 일을 목록으로 반환

[{'id': 1,
  'parent_id': None,
  'task': 'notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [5]:
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.

## Step 3: Add tools

A tool in Pydantic AI is a plain typed Python function. Pydantic reads the type hints and the docstring and builds the JSON schema for you, so there is nothing else to declare. The signature idiom is the `@agent.tool` decorator; passing the same functions in `tools=[...]` does exactly the same thing and lets us reuse them across agents, which is what we do here.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [6]:
def show_todos() -> list[dict]:
    """보드에 있는 모든 할 일을 나열합니다. 목표(goal)는 parent_id가 None이고, 단계(step)는 parent_id에 소속 목표의 id가 들어 있습니다."""
    return board.list_todos()  # 보드 모듈의 조회 함수를 그대로 호출해 결과를 반환


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """목표를 순서가 있는 단계 체크리스트로 나누어 보드에 등록합니다. 목표의 id와 짧은 단계 설명 목록을 전달하세요."""
    # steps의 각 문자열을 goal_id 아래 단계로 추가하고, 생성된 단계 id들을 리스트로 수집
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """이 id를 가진 할 일(단계 또는 목표)을 완료로 표시하고 짧은 결과 요약을 기록합니다."""
    board.complete_todo(task_id, result)  # 상태를 done으로 변경하고 result 문자열을 함께 저장
    return {"task_id": task_id, "status": "done"}  # 호출자에게 처리 결과를 딕셔너리로 반환

In [7]:
# 보드 관리용 에이전트 생성
board_agent = Agent(
    MODEL,  # 앞서 정의한 모델 식별자 사용
    instructions="당신은 공유 할 일 보드를 관리하는 것을 돕습니다.",  # 지속 적용되는 시스템 지시문
    tools=[show_todos, complete_task],  # 조회용과 완료 처리용 도구만 등록(plan_steps는 제외)
)

In [8]:
# 보드 에이전트 실행: 질문을 전달하면 에이전트가 show_todos 도구를 호출해 보드를 조회한 뒤 답변한다
result = await board_agent.run("지금 보드에 무엇이 있고, 그 상태는 어떻습니까?")
# 에이전트의 최종 응답 텍스트 출력
print(result.output)

보드에는 현재 1개의 할 일이 있습니다.

- **ID 1**: `notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.`
  - 상태: **pending** (미완료)

원하시면 제가 이 할 일의 진행 상태를 더 추적해드릴게요.


## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server, the same Node server every framework uses this week, scoped to a single `workspace` folder so it can only touch files in there. In Pydantic AI an MCP server is a toolset: you wrap it in an `MCPToolset`, pass it via `toolsets=[...]`, and open its connection with `async with agent:` around the run.

We point `log_file` at the null device (`Path(os.devnull)`). That keeps the server's startup logging off the screen and, more importantly, lets the server run from a Jupyter kernel on Windows, where the kernel's own stderr has no real file descriptor for the server to write to. On Mac and Linux it simply keeps the output clean.

In [9]:
# 에이전트가 접근할 수 있는 유일한 폴더. resolve()로 절대 경로로 변환
workspace = Path("workspace").resolve()

# 파일시스템 MCP 서버를 도구 모음으로 구성
filesystem = MCPToolset(
    StdioTransport(  # 표준입출력으로 MCP 서버와 통신하는 전송 방식
        command="npx",  # Node 패키지 실행기로 MCP 서버를 기동
        # -y: 설치 확인 프롬프트 생략, 마지막 인자로 접근 허용 디렉터리를 지정
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # 상대 경로 파일명이 workspace 기준으로 해석되도록 서버를 그 폴더에서 시작
        log_file=Path(os.devnull),  # 서버 로그를 버림(콘솔 출력 오염 방지)
    ),
    init_timeout=60  # MCP 서버 초기화 대기 시간(초)
)

In [10]:
# 파일 작업용 에이전트 생성
file_agent = Agent(
    MODEL,  # 앞서 정의한 모델 식별자 사용
    instructions="당신은 작업 폴더 안의 파일을 읽고 쓸 수 있습니다. 요청받은 일을 하기 위해 도구를 사용하세요.",  # 지속 적용되는 시스템 지시문
    toolsets=[filesystem],  # 파일 읽기/쓰기용 MCP 도구 모음을 연결
)

In [11]:
# async with 블록: MCP 서버 기동과 종료를 컨텍스트로 관리(블록을 벗어나면 서버가 정리됨)
async with file_agent:
    # 에이전트 실행: 파일 도구로 notes.txt를 읽은 뒤 요약을 생성한다
    result = await file_agent.run("notes.txt를 읽고 짧은 한 문장으로 요약해 주세요.")
# 에이전트의 최종 응답 텍스트 출력(블록 밖에서도 result 변수는 유지된다)
print(result.output)

팀원으로서 공유 보드의 각 작업을 하나씩 신중하게 수행하며 작은 언어 튜터를 함께 만들어 가자는 안내입니다.


## Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat. Watch the board fill with steps and then strike them through.

In [12]:
# 워커 에이전트에게 지속적으로 적용되는 시스템 지시문
INSTRUCTIONS = """
당신은 공유 할 일 보드와 파일 도구 모음을 갖춘 신중한 작업자입니다.

대기 중인 목표를 맡아 끝까지 완수하세요. 먼저 짧은 계획을 세우는 것으로 시작하세요: 그 작업이 실제로 나뉘는 몇 가지 구체적인 단계를 목표 아래 보드에 추가하세요. 그런 다음 파일 도구로 그 단계들을 수행하고, 각 단계를 끝낼 때마다 완료로 표시하세요. 단계가 모두 끝나면 목표를 닫으세요. 당신의 파일은 도구가 사용을 허가받은 단 하나의 폴더 안에 있습니다.
"""

# 워커 에이전트 생성: 보드 조작 도구와 파일 MCP 도구를 함께 갖춘 구성
worker = Agent(
    MODEL,  # 앞서 정의한 모델 식별자 사용
    instructions=INSTRUCTIONS,  # 위에서 정의한 시스템 지시문
    tools=[show_todos, plan_steps, complete_task],  # 보드 조회/계획/완료 처리용 도구 3종
    toolsets=[filesystem],  # 파일 읽기/쓰기용 MCP 도구 모음
)

board.reset_board()  # 보드의 기존 항목을 모두 제거해 빈 상태로 초기화
# 목표를 보드에 등록하고 생성된 id를 받음. 이 문자열은 에이전트가 읽어서 수행할 작업 지시문이다
goal_id = board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.")
board.claim_todo(goal_id)  # 워커가 목표를 점유: pending -> in_progress

# async with 블록: MCP 서버 기동과 종료를 컨텍스트로 관리
async with worker:
    # 에이전트 실행: 계획 수립 -> 파일 작업 -> 단계 완료 표시 -> 목표 닫기 순서로 진행된다
    result = await worker.run("보드에 대기 중인 목표를 수행해 주세요.")
print(result.output)  # 에이전트의 최종 응답 텍스트 출력
board.show_board()  # 실행 후 보드 전체 현황 출력

완료했습니다.  
`notes.txt`를 읽어 자연스러운 스페인어로 번역했고, `spanish.txt`에 저장한 뒤 확인까지 마쳤습니다.


Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 쓰세요.  notes.txt를 읽어 스페인어 번역을 spanish.txt에 저장하고 확인까지 완료했다.
  Step #2: 작업 폴더와 notes.txt 위치를 확인한다  작업 폴더에서 notes.txt 위치를 확인했다.
  Step #3: notes.txt 내용을 읽어 스페인어로 번역한다  notes.txt 내용을 읽고 자연스러운 스페인어로 번역했다.
  Step #4: 번역문을 spanish.txt에 저장한다  번역문을 spanish.txt에 저장했다.
  Step #5: 결과를 확인하고 목표를 완료한다  spanish.txt 내용을 확인했고 번역 파일이 정상적으로 저장되었음을 확인했다. 목표를 마무리했다.

## Run the same worker from the terminal

Everything you just watched in step 5 is also packaged as a small script, `pydantic_worker.py`, sitting next to this notebook. It seeds the same goal, builds the same agent with the same three board tools and the filesystem MCP server, and runs the same loop, only from the command line instead of the kernel. Open a terminal in this folder and run it:

```bash
uv run pydantic_worker.py
```

You will see the agent plan its steps, work the goal and tick each one off, then the finished board and the Spanish it wrote. This is the shape every worker takes on Day 5, where a Google ADK orchestrator launches one of these per framework as a parallel subprocess against a single shared board.

## The one cool thing: typed, validated, observable

Pydantic AI is built around two ideas worth knowing even though our worker does not lean on them:

- **Typed, validated results.** Pass a Pydantic model as `output_type=` and `result.output` comes back as a validated instance of that type. Every framework can do typed output; this is the thing Pydantic AI is known for.
- **Observability in two lines.** `logfire.configure()` and `logfire.instrument_pydantic_ai()` light up every model request, tool call and validation in the Logfire UI, live, from the same team that builds the framework. It needs the `logfire` extra, which we leave out to keep the shared environment lean.

And the same one-line spirit as the rest of the week: to run on any OpenAI-compatible endpoint, wrap an `OpenAIChatModel` around an `OpenAIProvider` and hand it to the agent. Everything else stays identical:

```python
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIChatModel(
    "gpt-5.4-mini",
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY),
)
agent = Agent(model, instructions="...")
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Seed a different goal on the board, for example "write a short haiku about Madrid into madrid.txt", and run the worker again. Does it plan sensible steps and pick the right file tools? Then, after a run, inspect <code>result.all_messages()</code> to see the full transcript: every tool call the loop made and every result it read back.</span>
        </td>
    </tr>
</table>